# Star Rating Simulation and Quality Bonus Payment Estimates

Lake DT, Civelek Y, Bond AM, Khullar D. Characteristics and financial implications of measures removed from Medicare Advantage star ratings. *JAMA Netw Open*. 2026;9(10):e2640240. doi:[10.1001/jamanetworkopen.2026.40240](https://doi.org/10.1001/jamanetworkopen.2026.40240)

Copyright 2026 Derek T. Lake. Code licensed under the Apache License, Version 2.0 (see `LICENSE` and `NOTICE`). Please cite the article above when using this code or its outputs.

This notebook replicates the 2026 Medicare Advantage (MA) Star Ratings published by the Centers for Medicare & Medicaid Services (CMS), recomputes each contract's ratings after removing the 13 measure components finalized for removal in April 2026, and prices the resulting change in quality bonus payment (QBP) eligibility. It produces the analytic cohort, the replication accuracy check, the rating changes, and the QBP estimates reported in the Research Letter and its eMethods. Notebook 02 reads its output.

**Contents**

1. Setup
2. Star rating algorithm
3. Data
4. Baseline ratings and replication accuracy
5. Disaster-affected contracts
6. Ratings after measure removal
7. Enrollment and QBP estimation
8. Analytic cohort
9. Output
10. Checks against the article

Run from the `notebooks` folder (the Jupyter default) or from the repository root. Inputs are read from `data/raw`; outputs are written to `data/derived`.

## 1. Setup

Paths are resolved relative to the repository root. Parameter values for the rating algorithm are taken from the 2026 Star Ratings Technical Notes; `docs/methods_sources.md` lists the source of each one.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_RAW = ROOT / "data" / "raw"
DATA_DERIVED = ROOT / "data" / "derived"
DATA_DERIVED.mkdir(parents=True, exist_ok=True)

# CMS 2026 Star Ratings Data Tables (October 8, 2025 release)
MEASURE_STARS_2026 = DATA_RAW / "2026 Star Ratings Data Table - Measure Stars (Oct 8 2025).csv"
SUMMARY_2026 = DATA_RAW / "2026 Star Ratings Data Table - Summary Ratings (Oct 8 2025).csv"
CAI_2026 = DATA_RAW / "2026 Star Ratings Data Table - CAI (Oct 8 2025).csv"
# CMS 2025 Star Ratings Data Tables (December 2, 2024 release), disaster diagnostic only
SUMMARY_2025 = DATA_RAW / "2025 Star Ratings Data Table - Summary Ratings (Dec 2 2024).csv"
# CMS Monthly MA Enrollment by State/County/Contract, May 2026
ENROLLMENT_SCC = DATA_RAW / "SCC_Enrollment_MA_2026_05.csv.zip"
# CMS 2026 MA ratebook, county capitation rates
RATEBOOK = DATA_RAW / "RateBook2026.xlsx"

In [2]:
# Measure weights, 2026 Star Ratings Technical Notes. Improvement measures C30 and D04 carry
# a weight of 5; all other weights are 1, 2, or 3.
MEASURE_WEIGHTS = {
    "C01": 1, "C02": 1, "C03": 1, "C04": 1, "C05": 1, "C06": 1, "C07": 1, "C08": 1,
    "C09": 1, "C10": 1, "C11": 1, "C12": 3, "C13": 1, "C14": 3, "C15": 1, "C16": 1,
    "C17": 1, "C18": 3, "C19": 1, "C20": 1, "C21": 1, "C22": 2, "C23": 2, "C24": 2,
    "C25": 2, "C26": 2, "C27": 2, "C28": 2, "C29": 2, "C30": 5, "C31": 2, "C32": 2,
    "C33": 2,
    "D01": 2, "D02": 2, "D03": 2, "D04": 5, "D05": 2, "D06": 2, "D07": 1, "D08": 3,
    "D09": 3, "D10": 3, "D11": 1, "D12": 1,
}

# Categorical adjustment index (CAI) values by final adjustment category, 2026 Star Ratings
# Technical Notes. Each contract's category is published in the CAI data table.
CAI_VALUES = {
    "Overall": {1: -0.063262, 2: -0.040422, 3: -0.017803, 4: 0.003256, 5: 0.018790,
                6: 0.045683, 7: 0.058145, 8: 0.101257, 9: 0.145515},
    "PartC": {1: -0.058259, 2: -0.036927, 3: -0.013699, 4: 0.004022, 5: 0.032302,
              6: 0.059788, 7: 0.080451, 8: 0.102370},
    "PartD_MAPD": {1: -0.033144, 2: -0.014987, 3: -0.002688, 4: 0.046282, 5: 0.072332,
                   6: 0.128476},
    "PartD_PDP": {1: -0.227881, 2: -0.082454, 3: 0.025549},
}

# Reward factor thresholds, 2026 Star Ratings Technical Notes, Tables 8 (mean) and 9 (variance).
# vl and vh are the 30th and 70th percentiles of contract variance; ml and mh are the 65th and
# 85th percentiles of the contract weighted mean. The tables are indexed by whether the rating
# includes the improvement measures and whether it includes new measures. CMS compares ratings
# with and without new measures only for disaster-affected contracts, which are excluded here,
# so both versions below include new measures: "with_imp" is the (Improvement: With, New
# Measures: With) row and "no_imp" is the (Improvement: Without, New Measures: With) row.
OFFICIAL_THRESHOLDS = {
    "Overall": {
        "with_imp": {"vl": 0.914850, "vh": 1.263462, "ml": 3.649351, "mh": 3.932432},
        "no_imp":   {"vl": 0.908919, "vh": 1.269610, "ml": 3.686567, "mh": 3.953125},
    },
    "PartC": {
        "with_imp": {"vl": 0.918435, "vh": 1.285170, "ml": 3.695652, "mh": 4.000000},
        "no_imp":   {"vl": 0.914326, "vh": 1.328432, "ml": 3.717391, "mh": 4.020408},
    },
    "PartD_MAPD": {
        "with_imp": {"vl": 0.754209, "vh": 1.268986, "ml": 3.740741, "mh": 4.000000},
        "no_imp":   {"vl": 0.736111, "vh": 1.318182, "ml": 3.769231, "mh": 4.136364},
    },
    "PartD_PDP": {
        "with_imp": {"vl": 0.869005, "vh": 1.747939, "ml": 3.385522, "mh": 3.913300},
        "no_imp":   {"vl": 0.749180, "vh": 1.814773, "ml": 3.318182, "mh": 4.117647},
    },
}

PART_C_MEASURES = [f"C{i:02d}" for i in range(1, 34)]
PART_D_MEASURES = [f"D{i:02d}" for i in range(1, 13)]
IMPROVEMENT_MEASURES = {"C30", "D04"}
SNP_ONLY_MEASURES = {"C07"}
# Complaints (D02) and Members Choosing to Leave the Plan (D03) duplicate C28 and C29 and are
# counted once, through the Part C version, in the overall rating of an MA-PD contract.
PART_D_DUPLICATES = {"D02", "D03"}

# The 13 measure components removed from the Star Ratings by the CMS final rule published
# April 6, 2026 (11 measures; Complaints and Members Choosing to Leave the Plan each have a
# Part C and a Part D component).
REMOVED_MEASURES = frozenset({
    "C07", "C19", "C24", "C25", "C28", "C29", "C31", "C32", "C33",
    "D01", "D02", "D03", "D07",
})

QBP_THRESHOLD = 4.0          # overall rating required for the 5% bonus benchmark (42 CFR 422.258)
DISASTER_THRESHOLD = 25.0    # percent of enrollment in FEMA-designated disaster areas
STAR_LEVELS = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]

## 2. Star rating algorithm

Each function below reproduces one step of the 2026 CMS methodology. A summary rating (Part C, Part D, or overall) is the weighted mean of the contract's measure-level stars, plus a reward factor for consistently high performance, plus the CAI value, rounded to the nearest half star. A rating requires the contract to have stars on at least half of the applicable non-improvement measures. Under the improvement measure hold harmless provision, a rating computed without the improvement measure replaces the rating with it when the rating without it is at least 4 stars and higher than the rating with it.

In [3]:
def round_half_star(x):
    """Round to the nearest half star; exact ties round up."""
    return np.floor(x * 2 + 0.5) / 2


def applicable_measures(part, is_snp, removed=frozenset()):
    """Measures that enter a summary rating, before checking which ones the contract reports."""
    part_c = [m for m in PART_C_MEASURES if is_snp or m not in SNP_ONLY_MEASURES]
    if part == "C":
        measures = part_c
    elif part == "D":
        measures = list(PART_D_MEASURES)
    else:
        measures = part_c + [m for m in PART_D_MEASURES if m not in PART_D_DUPLICATES]
    return [m for m in measures if m not in removed]


def weighted_mean(row, measures):
    vals = [(row[m], MEASURE_WEIGHTS[m]) for m in measures if pd.notna(row.get(m))]
    if not vals:
        return np.nan
    stars, weights = zip(*vals)
    return np.average(stars, weights=weights)


def weighted_variance(row, measures):
    """Weighted variance of measure stars as defined in the Technical Notes (improvement
    measures excluded). Contracts with fewer than 2 rated measures are assigned 0."""
    vals = [(row[m], MEASURE_WEIGHTS[m]) for m in measures
            if m not in IMPROVEMENT_MEASURES and pd.notna(row.get(m))]
    if len(vals) < 2:
        return 0.0
    stars, weights = map(np.array, zip(*vals))
    n, total_weight = len(stars), weights.sum()
    mean = np.average(stars, weights=weights)
    return n * np.sum(weights * (stars - mean) ** 2) / (total_weight * (n - 1))


def reward_factor(mean, variance, t):
    """Reward factor from the contract's weighted mean and variance relative to thresholds t."""
    if pd.isna(mean):
        return 0.0
    var_cat = "low" if variance < t["vl"] else ("medium" if variance < t["vh"] else "high")
    mean_cat = "high" if mean >= t["mh"] else ("relatively high" if mean >= t["ml"] else "other")
    table = {("high", "low"): 0.4, ("high", "medium"): 0.3,
             ("relatively high", "low"): 0.2, ("relatively high", "medium"): 0.1}
    return table.get((mean_cat, var_cat), 0.0)


def summary_statistics(row, part, removed=frozenset()):
    """Weighted means with and without the improvement measure, variance, and the minimum
    measure check for one contract and one summary rating."""
    measures = applicable_measures(part, row["Is_SNP"], removed)
    counted = [m for m in measures if m not in IMPROVEMENT_MEASURES]
    n_rated = sum(pd.notna(row.get(m)) for m in counted)
    return {
        "mean_wi": weighted_mean(row, measures),
        "mean_ni": weighted_mean(row, counted),
        "variance": weighted_variance(row, counted),
        "meets_min": n_rated >= (len(counted) + 1) // 2,
    }


def threshold_key(part, contract_type):
    if part == "C":
        return "PartC"
    if part == "D":
        return "PartD_PDP" if contract_type == "PDP" else "PartD_MAPD"
    return "Overall"


def cai_value(row, part):
    key = threshold_key(part, row["Contract_Type"])
    category_column = {"PartC": "FAC_C", "PartD_MAPD": "FAC_D_MAPD",
                       "PartD_PDP": "FAC_D_PDP", "Overall": "FAC_Overall"}[key]
    category = row.get(category_column)
    return CAI_VALUES[key].get(int(category), 0.0) if pd.notna(category) else 0.0


def summary_rating(row, part, stats, thresholds, hold_harmless):
    """One summary rating, applying the hold harmless provision when required."""
    if pd.isna(stats["mean_wi"]) or not stats["meets_min"]:
        return np.nan
    t = thresholds[threshold_key(part, row["Contract_Type"])]
    cai = cai_value(row, part)
    rating = min(5.0, round_half_star(
        stats["mean_wi"] + reward_factor(stats["mean_wi"], stats["variance"], t["with_imp"]) + cai))
    if hold_harmless and pd.notna(stats["mean_ni"]):
        rating_ni = min(5.0, round_half_star(
            stats["mean_ni"] + reward_factor(stats["mean_ni"], stats["variance"], t["no_imp"]) + cai))
        if rating_ni >= 4.0 and rating_ni > rating:
            rating = rating_ni
    return rating


def contract_ratings(row, thresholds, removed=frozenset()):
    """Part C, Part D, and overall ratings for one contract. The hold harmless provision
    applies to Part C for MA-only contracts, Part D for prescription drug plans, and the
    overall rating for MA-PD contracts. The overall rating requires both summary ratings."""
    ct = row["Contract_Type"]
    out = {"PartC": np.nan, "PartD": np.nan, "Overall": np.nan}
    if ct in ("MA-PD", "MA-Only"):
        out["PartC"] = summary_rating(row, "C", summary_statistics(row, "C", removed),
                                      thresholds, hold_harmless=(ct == "MA-Only"))
    if ct in ("MA-PD", "PDP"):
        out["PartD"] = summary_rating(row, "D", summary_statistics(row, "D", removed),
                                      thresholds, hold_harmless=(ct == "PDP"))
    if ct == "MA-PD" and pd.notna(out["PartC"]) and pd.notna(out["PartD"]):
        out["Overall"] = summary_rating(row, "Overall", summary_statistics(row, "Overall", removed),
                                        thresholds, hold_harmless=True)
    return out

## 3. Data

The Measure Stars table gives each contract's 1 to 5 star rating on every measure. The Summary Ratings table gives the CMS-published Part C, Part D, and overall ratings, the SNP flag, and the share of enrollment in FEMA-designated disaster areas in 2023 and 2024. The CAI table gives each contract's adjustment category. The CMS files carry a title row, and the Measure Stars table places its measure labels on the third row, so each reader below locates the header explicitly. Text entries such as "Plan too small to be measured" are treated as missing.

In [4]:
def read_measure_table(path):
    """Contract-by-measure table with columns Contract_ID, Organization_Type, and one
    column per measure code. Measure labels ("C01: Breast Cancer Screening") are on the
    third row; contract rows begin on the fifth."""
    raw = pd.read_csv(path, encoding="utf-8-sig", header=None, dtype=str, low_memory=False)
    labels = raw.iloc[2].tolist()
    columns = {0: "Contract_ID", 1: "Organization_Type"}
    for i, label in enumerate(labels):
        if isinstance(label, str) and ":" in label:
            columns[i] = label.split(":")[0].strip()
    table = raw.iloc[4:, list(columns)].rename(columns=columns)
    table["Contract_ID"] = table["Contract_ID"].str.strip()
    table = table[table["Contract_ID"].str.fullmatch(r"[A-Z0-9]{5}", na=False)]
    return table.reset_index(drop=True)


def to_number(series):
    """Numeric values; CMS text entries (for example, "Not enough data available") become missing."""
    return pd.to_numeric(series.str.strip().str.replace("%", "", regex=False), errors="coerce")


stars = read_measure_table(MEASURE_STARS_2026)
measure_codes = [c for c in stars.columns if c[:1] in ("C", "D") and len(c) == 3]
stars[measure_codes] = stars[measure_codes].apply(to_number)

summary = pd.read_csv(SUMMARY_2026, encoding="utf-8-sig", header=1, dtype=str).rename(columns={
    "Contract Number": "Contract_ID", "Organization Type": "Organization_Type",
    "2023 Disaster %": "Disaster_Pct_2023", "2024 Disaster %": "Disaster_Pct_2024",
    "2026 Part C Summary": "CMS_PartC", "2026 Part D Summary": "CMS_PartD",
    "2026 Overall": "CMS_Overall"})
summary["Contract_ID"] = summary["Contract_ID"].str.strip()
for col in ["CMS_PartC", "CMS_PartD", "CMS_Overall"]:
    summary[col] = to_number(summary[col])
for col in ["Disaster_Pct_2023", "Disaster_Pct_2024"]:
    summary[col] = to_number(summary[col]).fillna(0)

cai = pd.read_csv(CAI_2026, encoding="utf-8-sig", header=1, dtype=str).rename(columns={
    "Contract Number": "Contract_ID", "Part C FAC": "FAC_C", "Part D MA-PD FAC": "FAC_D_MAPD",
    "Part D PDP FAC": "FAC_D_PDP", "Overall FAC": "FAC_Overall"})
cai["Contract_ID"] = cai["Contract_ID"].str.strip()
fac_columns = ["FAC_C", "FAC_D_MAPD", "FAC_D_PDP", "FAC_Overall"]
cai[fac_columns] = cai[fac_columns].apply(to_number)

contracts = (stars.drop(columns="Organization_Type")
             .merge(summary[["Contract_ID", "Organization_Type", "SNP", "Disaster_Pct_2023",
                             "Disaster_Pct_2024", "CMS_PartC", "CMS_PartD", "CMS_Overall"]],
                    on="Contract_ID", how="left")
             .merge(cai[["Contract_ID"] + fac_columns], on="Contract_ID", how="left"))
print(f"Contracts in the 2026 Star Ratings: {len(contracts)}")

Contracts in the 2026 Star Ratings: 769


Contract type determines which summary ratings a contract receives. Prescription drug plans (PDPs) receive a Part D rating only, MA-only contracts a Part C rating only, and MA-PD contracts all three. Private fee-for-service and section 1876 cost contracts are classified by whether they report Part D measures. A contract is treated as a special needs plan (SNP) contract if the Summary Ratings table flags it or it reports the SNP Care Management measure.

In [5]:
def contract_type(row):
    org = str(row["Organization_Type"]).strip().lower()
    has_d = any(pd.notna(row.get(m)) for m in PART_D_MEASURES if m not in IMPROVEMENT_MEASURES)
    if "pdp" in org and "pffs" not in org and "ccp" not in org:
        return "PDP"
    if "pffs" in org or "1876" in org or "cost" in org:
        return "MA-PD" if any(pd.notna(row.get(m)) for m in PART_D_MEASURES) else "MA-Only"
    if "msa" in org:
        return "MA-Only"
    if "ccp" in org:
        return "MA-PD"
    has_c = any(pd.notna(row.get(m)) for m in PART_C_MEASURES if m not in IMPROVEMENT_MEASURES)
    if has_c and has_d:
        return "MA-PD"
    return "MA-Only" if has_c else ("PDP" if has_d else "Unknown")


contracts["Contract_Type"] = contracts.apply(contract_type, axis=1)
contracts["Is_SNP"] = (contracts["SNP"].str.strip().str.lower().eq("yes")
                       | contracts["C07"].notna())
print(contracts["Contract_Type"].value_counts().to_string())

Contract_Type
MA-PD      715
PDP         49
MA-Only      5


## 4. Baseline ratings and replication accuracy

Baseline ratings apply the CMS algorithm to all 2026 measures with the published reward factor thresholds. Agreement with the CMS-published overall rating is assessed among contracts that are not disaster affected (eMethods, Replication Accuracy). The simulated baseline rating, not the published rating, defines baseline QBP eligibility, so that baseline and projected ratings come from the same algorithm.

A contract has an overall rating in this analysis when both CMS and the simulation rate it. Sixteen small SNP contracts have a CMS-published overall rating but stars on fewer than half of the Part C measures that the simulation counts toward the minimum, so the simulation does not rate them. They are among the 269 contracts without an overall rating.

In [6]:
baseline = pd.DataFrame([contract_ratings(row, OFFICIAL_THRESHOLDS)
                         for _, row in contracts.iterrows()])
contracts[["Baseline_PartC", "Baseline_PartD", "Baseline_Overall"]] = baseline[
    ["PartC", "PartD", "Overall"]].to_numpy()

contracts["Disaster_Affected"] = ((contracts["Disaster_Pct_2023"] >= DISASTER_THRESHOLD)
                                  | (contracts["Disaster_Pct_2024"] >= DISASTER_THRESHOLD))
contracts["Has_Overall"] = contracts["CMS_Overall"].notna() & contracts["Baseline_Overall"].notna()

replication = contracts[contracts["Has_Overall"] & ~contracts["Disaster_Affected"]]
difference = replication["Baseline_Overall"] - replication["CMS_Overall"]
contracts["Replication_Match"] = np.where(contracts["Has_Overall"],
                                          contracts["Baseline_Overall"] == contracts["CMS_Overall"],
                                          np.nan)
n_mismatch = int((difference != 0).sum())
n_cms_only = int((contracts["CMS_Overall"].notna() & contracts["Baseline_Overall"].isna()).sum())
print(f"Contracts with a CMS overall rating that the simulation does not rate: {n_cms_only}")
print(f"Non-disaster contracts with CMS and simulated overall ratings: {len(replication)}")
print(f"Overall rating differs from CMS: {n_mismatch} ({n_mismatch / len(replication):.1%})")
print(f"Absolute differences among mismatches: {sorted(set(difference[difference != 0].abs()))}")

Contracts with a CMS overall rating that the simulation does not rate: 16
Non-disaster contracts with CMS and simulated overall ratings: 436
Overall rating differs from CMS: 5 (1.1%)
Absolute differences among mismatches: [0.5]


## 5. Disaster-affected contracts

CMS applies an extreme and uncontrollable circumstances policy to contracts with at least 25% of enrollment in FEMA-designated disaster areas: for affected measures, the contract receives the higher of its current-year and prior-year measure stars. The measure-level inputs to that policy are not published, so these contracts are excluded. The diagnostic below compares, for affected contracts whose simulated rating differs from the published one, the published rating with the higher of the simulated rating and the published 2025 rating. It checks only the direction of the discrepancy at the summary level; the policy itself cannot be reproduced from public data.

In [7]:
summary_2025 = pd.read_csv(SUMMARY_2025, encoding="utf-8", header=1, dtype=str).rename(columns={
    "Contract Number": "Contract_ID", "2025 Overall": "CMS_2025_Overall",
    "2025 Part C Summary": "CMS_2025_PartC", "2025 Part D Summary": "CMS_2025_PartD"})
summary_2025["Contract_ID"] = summary_2025["Contract_ID"].str.strip()
for col in ["CMS_2025_Overall", "CMS_2025_PartC", "CMS_2025_PartD"]:
    summary_2025[col] = to_number(summary_2025[col])

parts = ["Overall", "PartC", "PartD"]
disaster = contracts[contracts["Disaster_Affected"]].merge(
    summary_2025[["Contract_ID"] + [f"CMS_2025_{p}" for p in parts]], on="Contract_ID", how="left")
mismatch = pd.concat([(disaster[f"Baseline_{p}"] != disaster[f"CMS_{p}"])
                      & disaster[f"Baseline_{p}"].notna() & disaster[f"CMS_{p}"].notna()
                      for p in parts], axis=1).any(axis=1)
diagnostic = disaster.loc[mismatch, ["Contract_ID", "Disaster_Pct_2023", "Disaster_Pct_2024"]].copy()
for p in parts:
    d = disaster.loc[mismatch]
    diagnostic[f"Baseline_{p}"] = d[f"Baseline_{p}"]
    diagnostic[f"CMS_2025_{p}"] = d[f"CMS_2025_{p}"]
    diagnostic[f"Higher_Of_{p}"] = d[[f"Baseline_{p}", f"CMS_2025_{p}"]].max(axis=1)
    diagnostic[f"CMS_{p}"] = d[f"CMS_{p}"]
diagnostic.to_csv(DATA_DERIVED / "disaster_mismatch_diagnostic.csv", index=False)

n_disaster_rated = int((contracts["Disaster_Affected"] & contracts["Has_Overall"]).sum())
print(f"Disaster-affected contracts with CMS and simulated overall ratings: {n_disaster_rated}")
print(f"  qualifying on 2024 enrollment: "
      f"{int((contracts['Disaster_Affected'] & contracts['Has_Overall'] & (contracts['Disaster_Pct_2024'] >= DISASTER_THRESHOLD)).sum())}")
print(f"Disaster-affected contracts with any simulated rating that differs from CMS: {len(diagnostic)}")

Disaster-affected contracts with CMS and simulated overall ratings: 64
  qualifying on 2024 enrollment: 64
Disaster-affected contracts with any simulated rating that differs from CMS: 6


## 6. Ratings after measure removal

Projected ratings drop the 13 removed components from every summary rating. Measure-level stars are unchanged, because CMS assigns them measure by measure. The minimum measure requirement is reapplied to the reduced measure set. Reward factor thresholds are percentiles of the distribution of contract means and variances, so they are recalculated from the post-removal distribution using the same percentiles (eMethods, Reward Factor Recalibration). As in the published thresholds, each distribution includes every contract of the relevant type with a computable mean. The CAI value is held at its published 2026 level.

In [8]:
PERCENTILES = {"ml": 65, "mh": 85, "vl": 30, "vh": 70}
RATING_GROUPS = {  # threshold key -> (summary rating, contract types in the distribution)
    "PartC": ("C", ("MA-PD", "MA-Only")),
    "PartD_MAPD": ("D", ("MA-PD",)),
    "PartD_PDP": ("D", ("PDP",)),
    "Overall": ("Overall", ("MA-PD",)),
}

recalibrated, threshold_rows = {}, []
for key, (part, types) in RATING_GROUPS.items():
    group = contracts[contracts["Contract_Type"].isin(types)]
    stats = pd.DataFrame([summary_statistics(row, part, REMOVED_MEASURES)
                          for _, row in group.iterrows()])
    recalibrated[key] = {}
    for version, mean_col in [("with_imp", "mean_wi"), ("no_imp", "mean_ni")]:
        valid = stats.dropna(subset=[mean_col, "variance"])
        t = {"ml": np.percentile(valid[mean_col], PERCENTILES["ml"]),
             "mh": np.percentile(valid[mean_col], PERCENTILES["mh"]),
             "vl": np.percentile(valid["variance"], PERCENTILES["vl"]),
             "vh": np.percentile(valid["variance"], PERCENTILES["vh"])}
        recalibrated[key][version] = t
        for name in ["vl", "vh", "ml", "mh"]:
            threshold_rows.append({"Rating": key, "Version": version, "Threshold": name,
                                   "Percentile": PERCENTILES[name],
                                   "Published_2026": OFFICIAL_THRESHOLDS[key][version][name],
                                   "Recalibrated": t[name], "N_Contracts": len(valid)})

thresholds_table = pd.DataFrame(threshold_rows)
thresholds_table.to_csv(DATA_DERIVED / "reward_factor_thresholds.csv", index=False)

projected = pd.DataFrame([contract_ratings(row, recalibrated, REMOVED_MEASURES)
                          for _, row in contracts.iterrows()])
contracts[["Projected_PartC", "Projected_PartD", "Projected_Overall"]] = projected[
    ["PartC", "PartD", "Overall"]].to_numpy()
thresholds_table.round(4)

,Rating,Version,Threshold,Percentile,Published_2026,Recalibrated,N_Contracts
0,PartC,with_imp,vl,30,0.9184,0.7931,584
1,PartC,with_imp,vh,70,1.2852,1.2567,584
2,PartC,with_imp,ml,65,3.6957,3.6571,584
3,PartC,with_imp,mh,85,4.0000,3.9552,584
4,PartC,no_imp,vl,30,0.9143,0.7931,584
5,PartC,no_imp,vh,70,1.3284,1.2567,584
6,PartC,no_imp,ml,65,3.7174,3.6667,584
7,PartC,no_imp,mh,85,4.0204,4.0000,584
8,PartD_MAPD,with_imp,vl,30,0.7542,0.4356,598
9,PartD_MAPD,with_imp,vh,70,1.2690,0.9126,598


## 7. Enrollment and QBP estimation

Contract enrollment is the sum of county enrollment in the May 2026 State/County/Contract file. CMS suppresses counts of 10 or fewer enrollees; these are counted as 0. Five cohort contracts are listed in the file with every county count suppressed and therefore carry 0 enrollment and 0 QBP value. For each county, the annual value of the QBP is the difference between the Parts A and B 5% bonus benchmark rate and the 0% bonus benchmark rate in the 2026 ratebook, multiplied by May 2026 enrollment and by 12 months (eMethods, Quality Bonus Payment Calculation). County values are summed to the contract. A contract receives its QBP value when its overall rating is at least 4.0 stars.

In [9]:
scc = pd.read_csv(ENROLLMENT_SCC, dtype=str)
scc["Contract_ID"] = scc["Contract ID"].str.strip()
scc["SSA_Code"] = scc["SSA Code"].str.strip().str.zfill(5)
# Suppressed county counts (10 or fewer enrollees) appear as "." and are counted as 0.
scc["Enrolled"] = pd.to_numeric(scc["Enrolled"].str.strip().str.replace(",", "", regex=False),
                                errors="coerce").fillna(0)

rates = pd.read_excel(RATEBOOK, sheet_name="CountyRate2026", header=2)
rates.columns = rates.columns.str.strip().str.replace("\n", " ")
rates = rates.rename(columns={"Code": "SSA_Code",
                              "Parts A&B 5% Bonus 2026 Rate": "Rate_5pct",
                              "Parts A&B 0% Bonus 2026 Rate": "Rate_0pct"})
rates["SSA_Code"] = rates["SSA_Code"].astype(str).str.zfill(5)
for col in ["Rate_5pct", "Rate_0pct"]:
    rates[col] = pd.to_numeric(rates[col].astype(str).str.replace(",", ""), errors="coerce")

county = (scc[scc["Enrolled"] > 0]
          .merge(rates[["SSA_Code", "Rate_5pct", "Rate_0pct"]], on="SSA_Code", how="left")
          .dropna(subset=["Rate_5pct", "Rate_0pct"]))
county["Benchmark_5pct"] = county["Enrolled"] * county["Rate_5pct"] * 12
county["Benchmark_0pct"] = county["Enrolled"] * county["Rate_0pct"] * 12

benchmarks = county.groupby("Contract_ID")[["Benchmark_5pct", "Benchmark_0pct"]].sum()
enrollment = scc.groupby("Contract_ID")["Enrolled"].sum().rename("Enrollment")

contracts = contracts.merge(enrollment, left_on="Contract_ID", right_index=True, how="left")
contracts = contracts.merge(benchmarks, left_on="Contract_ID", right_index=True, how="left")
# Contracts listed in the enrollment file with every county count suppressed have no
# reportable enrollment, so their benchmark payments and QBP value are 0.
listed = contracts["Enrollment"].notna()
for col in ["Benchmark_5pct", "Benchmark_0pct"]:
    contracts.loc[listed, col] = contracts.loc[listed, col].fillna(0.0)
contracts["Annual_QBP_Value"] = contracts["Benchmark_5pct"] - contracts["Benchmark_0pct"]

## 8. Analytic cohort

Exclusions are applied in the order reported in the eMethods (Sample Exclusions). Contracts without linkable enrollment do not appear in the May 2026 enrollment file.

In [10]:
def exclusion_reason(row):
    if not row["Has_Overall"]:
        return "No overall rating"
    if row["Disaster_Affected"]:
        return "Disaster affected"
    if pd.isna(row["Projected_Overall"]):
        return "Below minimum measure threshold after removal"
    if pd.isna(row["Enrollment"]):
        return "No linkable May 2026 enrollment"
    return "Included"


contracts["Exclusion_Reason"] = contracts.apply(exclusion_reason, axis=1)
contracts["In_Analytic_Cohort"] = contracts["Exclusion_Reason"].eq("Included")
cohort = contracts["In_Analytic_Cohort"]

contracts["Rating_Change"] = np.where(cohort, contracts["Projected_Overall"] - contracts["Baseline_Overall"], np.nan)
contracts["Baseline_QBP_Eligible"] = np.where(cohort, contracts["Baseline_Overall"] >= QBP_THRESHOLD, np.nan)
contracts["Projected_QBP_Eligible"] = np.where(cohort, contracts["Projected_Overall"] >= QBP_THRESHOLD, np.nan)
contracts["QBP_Change"] = np.select(
    [cohort & (contracts["Baseline_Overall"] >= QBP_THRESHOLD) & (contracts["Projected_Overall"] < QBP_THRESHOLD),
     cohort & (contracts["Baseline_Overall"] < QBP_THRESHOLD) & (contracts["Projected_Overall"] >= QBP_THRESHOLD),
     cohort & (contracts["Baseline_Overall"] >= QBP_THRESHOLD) & (contracts["Projected_Overall"] >= QBP_THRESHOLD),
     cohort],
    ["Lose", "Gain", "Retain", "Never"], default="")
contracts["Baseline_QBP"] = np.where(cohort, np.where(contracts["Baseline_Overall"] >= QBP_THRESHOLD,
                                                      contracts["Annual_QBP_Value"], 0.0), np.nan)
contracts["Projected_QBP"] = np.where(cohort, np.where(contracts["Projected_Overall"] >= QBP_THRESHOLD,
                                                       contracts["Annual_QBP_Value"], 0.0), np.nan)
contracts["Net_QBP_Change"] = contracts["Projected_QBP"] - contracts["Baseline_QBP"]

EXCLUSIONS = ["No overall rating", "Disaster affected",
              "Below minimum measure threshold after removal", "No linkable May 2026 enrollment"]
steps = [("All contracts in the 2026 Star Ratings", len(contracts))]
for reason in EXCLUSIONS:
    steps.append((f"Excluded: {reason}", -int((contracts["Exclusion_Reason"] == reason).sum())))
steps.append(("Analytic cohort", int(cohort.sum())))
sample = pd.DataFrame(steps, columns=["Step", "Contracts"])
sample["Enrollees"] = [contracts["Enrollment"].sum()] + [
    -contracts.loc[contracts["Exclusion_Reason"] == r, "Enrollment"].sum() for r in EXCLUSIONS
] + [contracts.loc[cohort, "Enrollment"].sum()]
sample["Enrollees"] = sample["Enrollees"].astype(int)
sample.to_csv(DATA_DERIVED / "sample_construction.csv", index=False)
sample

,Step,Contracts,Enrollees
0,All contracts in the 2026 Star Ratings,769,35826318
1,Excluded: No overall rating,-269,-1049448
2,Excluded: Disaster affected,-64,-5689901
3,Excluded: Below minimum measure threshold afte...,-6,-79005
4,Excluded: No linkable May 2026 enrollment,-5,0
5,Analytic cohort,425,29007964


## 9. Output

`contract_level_results.csv` holds one row per contract in the 2026 Star Ratings, with ratings, enrollment, QBP values, and the exclusion reason. `measure_weights.csv` records the weight and removal status of each measure for use in notebook 02. Variables are defined in `docs/data_dictionary.md`.

In [11]:
OUTPUT_COLUMNS = [
    "Contract_ID", "Organization_Type", "Contract_Type", "Is_SNP",
    "Disaster_Pct_2023", "Disaster_Pct_2024", "Disaster_Affected",
    "CMS_PartC", "CMS_PartD", "CMS_Overall",
    "Baseline_PartC", "Baseline_PartD", "Baseline_Overall", "Replication_Match",
    "Projected_PartC", "Projected_PartD", "Projected_Overall", "Rating_Change",
    "Enrollment", "Benchmark_5pct", "Benchmark_0pct", "Annual_QBP_Value",
    "Baseline_QBP_Eligible", "Projected_QBP_Eligible", "QBP_Change",
    "Baseline_QBP", "Projected_QBP", "Net_QBP_Change",
    "Exclusion_Reason", "In_Analytic_Cohort",
]
results = contracts[OUTPUT_COLUMNS].copy()
results["Organization_Type"] = results["Organization_Type"].str.strip()
for col in ["Replication_Match", "Baseline_QBP_Eligible", "Projected_QBP_Eligible"]:
    results[col] = results[col].map({1.0: True, 0.0: False})
results.to_csv(DATA_DERIVED / "contract_level_results.csv", index=False)
print(f"Wrote {len(results)} contracts to data/derived/contract_level_results.csv")

measure_parameters = pd.DataFrame({
    "Measure": list(MEASURE_WEIGHTS),
    "Part": ["C" if m.startswith("C") else "D" for m in MEASURE_WEIGHTS],
    "Weight": list(MEASURE_WEIGHTS.values()),
    "Improvement_Measure": [m in IMPROVEMENT_MEASURES for m in MEASURE_WEIGHTS],
    "Removed": [m in REMOVED_MEASURES for m in MEASURE_WEIGHTS],
})
measure_parameters.to_csv(DATA_DERIVED / "measure_weights.csv", index=False)

Wrote 769 contracts to data/derived/contract_level_results.csv


## 10. Checks against the article

Each assertion restates a number reported in the Research Letter or its eMethods. A failed assertion stops the notebook.

In [12]:
c = results[results["In_Analytic_Cohort"]]
reasons = results["Exclusion_Reason"].value_counts()

# eMethods, Sample Exclusions
assert len(results) == 769
assert reasons["No overall rating"] == 269
assert reasons["Disaster affected"] == 64
assert reasons["Below minimum measure threshold after removal"] == 6
assert reasons["No linkable May 2026 enrollment"] == 5
assert len(c) == 425
assert c["Enrollment"].sum() == 29_007_964

# eMethods, Replication Accuracy: 5 of 436 non-disaster contracts, each off by 0.5 stars
assert len(replication) == 436 and n_mismatch == 5
# Documented in section 4: CMS-rated contracts the simulation does not rate (within the 269)
assert n_cms_only == 16
assert set(difference[difference != 0].abs()) == {0.5}

# Results: rating changes
change = c["Rating_Change"]
assert round(100 * (change < 0).mean(), 1) == 35.1
assert round(100 * (change == 0).mean(), 1) == 62.6
assert round(100 * (change > 0).mean(), 1) == 2.4
assert round(c["Baseline_Overall"].mean(), 2) == 3.58
assert round(c["Projected_Overall"].mean(), 2) == 3.41

# Results and Table: QBP eligibility and payments
groups = c.groupby("QBP_Change")
assert groups.size().to_dict() == {"Lose": 33, "Gain": 6, "Retain": 113, "Never": 273}
assert groups["Enrollment"].sum()["Lose"] == 2_587_448
assert groups["Enrollment"].sum()["Gain"] == 239_560
assert c["Annual_QBP_Value"].notna().all()
assert round(c["Baseline_QBP"].sum() / 1e6) == 15_057
assert round(c["Net_QBP_Change"].sum() / 1e6) == -2_093
assert round(100 * c["Net_QBP_Change"].sum() / c["Baseline_QBP"].sum(), 1) == -13.9
print("All notebook 01 checks passed.")

All notebook 01 checks passed.
